# W12. 여러 변수를 통제하면 결과가 달라지는가? — 다중회귀와 로지스틱 회귀 초급

> 다중회귀를 실행하고 잔차 진단을 수행할 수 있으며, 이분형 종속변수에는 로지스틱 회귀를 초급 수준에서 적용할 수 있는가?

**학습 목표**
1. 본인 연구 질문에 다중회귀를 적용하고, **잔차 진단**을 통해 모형의 적절성을 확인할 수 있다.
2. 이분형 종속변수(예: 투표 여부, 취업 여부)에는 **로지스틱 회귀 초급**을 적용할 수 있다.
3. 분석 전 과정을 **재현 가능 노트북**으로 정리하고, 보고서 본문 초안 4쪽을 작성할 수 있다.

In [ ]:
# 크로스플랫폼 한글 폰트 설정 (macOS / Windows)
import platform
import matplotlib.pyplot as plt

if platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
elif platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

## A회차: 회귀 선택과 잔차 진단 (방법론 보강)

### 내 종속변수는 어떤 회귀를 부르는가

| 종속변수 유형 | 추천 회귀 | 근거 |
|---|---|---|
| 연속형 (소득, 점수 등) | **다중회귀 (OLS)** | 7-8주에서 학습 |
| 이분형 (O/X, 유/무) | **로지스틱 회귀** | 오늘 초급 학습 |
| 3+ 범주 (지지정당 등) | (본 과목 범위 밖) | 보고서에서는 이분형으로 재범주화 권장 |

**원칙**: 종속변수에 맞지 않는 회귀를 쓰면 결과가 왜곡된다. 이분형 변수에 OLS를 돌리는 것은 흔한 실수다.

### 로지스틱 회귀 초급 — 딱 필요한 만큼만

학부 수준에서 이해해야 할 핵심 3가지:

1. **종속변수가 확률 logit**: `logit(p) = β0 + β1X1 + ...`
2. **계수 해석**: `exp(β1)`이 **오즈비(Odds Ratio)**. 1보다 크면 증가, 작으면 감소.
3. **p값 해석은 OLS와 동일**: 유의수준 하에서 계수가 0이 아니라고 할 수 있는가.

**Python 코드 템플릿**:
```python
import statsmodels.formula.api as smf
import numpy as np

model = smf.logit('졸업여부 ~ 부모학력 + 성별 + 연령', data=df).fit()
print(model.summary())

# 오즈비 + 95% CI
params = model.params
ci = model.conf_int()
odds = np.exp(pd.concat([params, ci], axis=1))
odds.columns = ['OR', 'CI_low', 'CI_high']
print(odds)
```

**해석 예시**: "부모 학력이 1단계 높을수록, 자녀의 대학 졸업 오즈가 약 1.45배 증가한다(OR=1.45, 95% CI: 1.12-1.87, p<0.01). 단, 이는 인과관계를 의미하지 않는다."

### 가상 예시: 대학 졸업 여부 분석

**연구 질문**: "부모의 학력과 가구소득이 자녀의 대학 졸업 여부에 영향을 미치는가?"

**변수 설정**:
- 종속변수: `대학졸업` (0=미졸업, 1=졸업)
- 독립변수: `부모학력` (1=중졸 이하, 2=고졸, 3=전문대졸, 4=대졸 이상)
- 통제변수: `가구소득_log` (로그 변환), `성별` (0=남성, 1=여성), `수도권` (0=비수도권, 1=수도권)

**가상 분석 결과**:

| 변수 | 계수(β) | SE | p값 | OR | 95% CI |
|---|---:|---:|---:|---:|---|
| 절편 | -2.10 | 0.45 | <0.001 | 0.12 | - |
| 부모학력 | 0.52 | 0.15 | <0.001 | **1.68** | 1.25-2.26 |
| 가구소득_log | 0.38 | 0.12 | 0.002 | **1.46** | 1.15-1.85 |
| 성별(여성=1) | -0.15 | 0.18 | 0.403 | 0.86 | 0.60-1.23 |
| 수도권(=1) | 0.41 | 0.16 | 0.010 | **1.51** | 1.10-2.07 |

Pseudo R² = 0.18, N = 850

**단계별 해석**:

1. **부모학력 (OR=1.68, p<0.001)**: 
   - "부모의 학력이 1단계 높아질수록(예: 고졸→전문대졸), 자녀가 대학을 졸업할 오즈가 약 1.68배 증가한다."
   - 95% CI가 1을 포함하지 않으므로 통계적으로 유의하다.
   - **효과크기**: (1.68-1)×100 = 68% 증가 → 실질적으로 큰 효과

2. **가구소득_log (OR=1.46, p=0.002)**:
   - "가구소득이 1% 증가할 때, 대학 졸업 오즈가 약 1.46배 증가한다."
   - 통계적으로 유의하며, 중간 수준의 효과

3. **성별 (OR=0.86, p=0.403)**:
   - "여성이 남성에 비해 대학 졸업 오즈가 약 0.86배(14% 감소)지만, 통계적으로 유의하지 않다(p=0.403)."
   - → 본 표본에서는 성별 차이를 확인할 수 없다.

4. **수도권 (OR=1.51, p=0.010)**:
   - "수도권 거주자가 비수도권에 비해 대학 졸업 오즈가 약 1.51배 높다."
   - 통계적으로 유의

**종합 해석**: "부모학력과 가구소득이 높을수록, 그리고 수도권에 거주할수록 자녀의 대학 졸업 확률이 높아진다. 단, 성별 차이는 본 표본에서 통계적으로 유의하지 않았다. 이는 상관관계이며, 인과관계를 주장하려면 추가적인 설계(예: 매개분석, 종단연구)가 필요하다."

**한계**: 
- 생략변수 편향 가능성 (학생 본인의 학업동기, 건강상태 등 미통제)
- Pseudo R²가 0.18로 낮은 편 → 설명되지 않은 변동이 크다

### 다중회귀(OLS) 잔차 진단 체크리스트

OLS 다중회귀를 돌린 뒤 반드시 확인한다.

| 가정 | 확인 방법 | 위배 시 |
|---|---|---|
| **선형성** | 잔차 vs 예측값 산점도 | 변수 변환 (로그 등) 고려 |
| **등분산성** | 잔차 vs 예측값 산점도의 퍼짐 | 이분산 견고 표준오차 (HC3) |
| **정규성** | 잔차의 Q-Q plot | 대표본(N≥200)이면 크게 문제 없음 |
| **다중공선성** | VIF < 10 | 상관 높은 변수 제거 또는 결합 |
| **독립성** | 잔차의 자기상관 (시계열일 때) | 본 과목 범위 밖 |

**원칙**: 가정 위배를 **숨기지 않는다**. 보고서 한계 섹션에 적는다.

### 가상 예시: 소득 결정요인 다중회귀 분석

**연구 질문**: "학력과 경력이 개인 소득에 미치는 영향은 무엇인가?"

**변수 설정**:
- 종속변수: `월소득` (만원, 연속형)
- 독립변수: `학력년수` (6~20년), `경력년수` (0~40년)
- 통제변수: `성별` (0=남성, 1=여성), `수도권` (0=비수도권, 1=수도권), `산업_제조업` (더미)

**가상 분석 결과**:

| 변수 | 계수(β) | SE | t | p값 | 95% CI |
|---|---:|---:|---:|---:|---|
| 절편 | 52.3 | 18.4 | 2.84 | 0.005 | 16.2-88.4 |
| 학력년수 | 18.5 | 2.1 | 8.81 | <0.001 | **14.4-22.6** |
| 경력년수 | 8.2 | 1.3 | 6.31 | <0.001 | **5.7-10.7** |
| 성별(여성=1) | -45.2 | 12.5 | -3.62 | <0.001 | **-69.7--20.7** |
| 수도권(=1) | 32.1 | 11.8 | 2.72 | 0.007 | **9.0-55.2** |
| 산업_제조업 | 15.3 | 14.2 | 1.08 | 0.282 | -12.6-43.2 |

R² = 0.42, Adj. R² = 0.41, F(5, 744) = 107.3, p < 0.001, N = 750

**단계별 해석**:

1. **학력년수 (β=18.5, p<0.001)**:
   - "학력이 1년 증가할 때마다, 월소득이 평균 18.5만원 증가한다."
   - 예: 고졸(12년) → 대졸(16년) = 18.5 × 4 = **74만원** 차이
   - 통계적으로 매우 유의하며, 실질적으로도 큰 효과

2. **경력년수 (β=8.2, p<0.001)**:
   - "경력이 1년 증가할 때마다, 월소득이 평균 8.2만원 증가한다."
   - 통계적으로 유의

3. **성별 (β=-45.2, p<0.001)**:
   - "여성은 남성에 비해 월소득이 평균 45.2만원 낮다 (다른 조건 동일 시)."
   - 성별 임금 격차가 통계적으로 유의 → 한계 섹션에서 논의 필요

4. **수도권 (β=32.1, p=0.007)**:
   - "수도권 거주자는 비수도권에 비해 월소득이 평균 32.1만원 높다."

5. **산업_제조업 (β=15.3, p=0.282)**:
   - "제조업 종사자는 다른 산업에 비해 월소득이 평균 15.3만원 높지만, 통계적으로 유의하지 않다(p=0.282)."
   - → 본 표본에서는 산업 간 차이를 확인할 수 없음

**모형 전체 평가**:
- R² = 0.42 → 소득 변동의 42%를 설명 (사회과학 기준 양호)
- F검정 p < 0.001 → 모형이 통계적으로 유의

---

**잔차 진단 결과**:

#### ① 등분산성 확인: 잔차의 퍼짐

**판단**: 예측값이 커질수록 잔차의 퍼짐이 약간 증가하는 경향 (경미한 이분산성) → 로버스트 표준오차(HC3) 사용 권장

```python
# 이분산 견고 표준오차 적용
from statsmodels.regression.linear_model import OLS
model_robust = OLS(y, X).fit(cov_type='HC3')
print(model_robust.summary())
```

#### ② 다중공선성 확인: VIF

```python
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_data = pd.DataFrame()
vif_data["변수"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]
print(vif_data)
```

**결과**:

| 변수 | VIF |
|---|---:|
| 학력년수 | 2.1 |
| 경력년수 | 1.8 |
| 성별 | 1.3 |
| 수도권 | 1.5 |
| 산업_제조업 | 1.4 |

**판단**: 모든 VIF < 10 (심지어 < 3) → 다중공선성 문제 없음 ✓

---

**최종 보고서 서술 예시**:

"학력년수와 경력년수가 월소득에 유의한 정(+)의 영향을 미치는 것으로 나타났다(각각 β=18.5, p<0.001; β=8.2, p<0.001). 성별과 수도권 거주 여부도 통계적으로 유의했으나, 산업(제조업 더미)은 유의하지 않았다. 모형은 소득 변동의 42%를 설명했다(R²=0.42). 잔차 진단 결과, 다중공선성 문제는 없었으나(모든 VIF < 3), 경미한 이분산성이 관찰되어 로버스트 표준오차(HC3)를 적용했다."

**한계**:
- 생략변수 편향 가능성 (직무 특성, 기업 규모 등 미포함)
- 경미한 이분산성 → HC3 표준오차로 보정
- 횡단면 데이터로 인과관계 단언 불가

### 재현성 — 노트북 품질 점검

보고서와 함께 제출할 노트북이 갖춰야 할 것:

1. **셀 순서 = 실행 순서**: 위에서 아래로 한 번에 실행되어야 한다.
2. **데이터 경로는 상대 경로**: `../data/KGSS2023.csv`처럼.
3. **랜덤 시드**: 무작위 요소가 있다면 `np.random.seed(42)`.
4. **환경 기록**: 첫 셀에 `!pip freeze | grep -E 'pandas|scipy|statsmodels'`.
5. **중간 출력 유지**: 주요 단계마다 `print()` 또는 `display()`.

### Copilot 활용 안내

**추천 프롬프트**:
- "이 OLS 모형의 잔차 vs 예측값 산점도와 Q-Q plot을 그리는 코드를 만들어줘."
- "변수 A, B, C, D의 VIF를 계산하는 코드를 만들어줘."
- "이분형 변수 [Y]에 대해 [X1, X2, X3]를 사용한 로지스틱 회귀를 돌리고, 오즈비와 95% CI를 함께 출력해줘."

## B회차: 개인 체크포인트 3 — 과제 12

### 과제 안내

| 항목 | 내용 |
|---|---|
| 과제 유형 | **개인 과제** |
| 소요 시간 | 40분 |
| 제출 형식 | 다중회귀(또는 로지스틱) 결과 + 잔차 진단 + 본문 초안 4쪽 |
| 제출처 | Google Classroom |

### 과제 구성

**제출 템플릿**: `practice/brief/cp3_regression.md`을 복사하여 작성한다. 본문 초안 4쪽은 `research_brief_template.md`(또는 `.docx`)의 1-4절을 사용한다.

#### ① 다중회귀 또는 로지스틱 회귀 실행

- 종속변수 유형에 맞는 회귀 선택
- 독립변수 + 통제변수 ≥ 2개 포함
- 계수 추정 결과표 (β, SE, p값, 95% CI)
- 로지스틱일 경우 오즈비 포함

#### ② 잔차 진단 (OLS) 또는 적합도 확인 (로지스틱)

- OLS: 잔차 vs 예측값, Q-Q plot, VIF
- 로지스틱: Pseudo R², 분류표 또는 Hosmer-Lemeshow
- 가정 위배 여부와 본인 판단

#### ③ 본문 초안 4쪽

아래 4개 섹션을 초안 수준으로 작성한다.

1. **서론 (0.5쪽)**: 연구 질문 + 중요성 + 선행연구 간략 언급
2. **선행연구 간략 검토 (1쪽)**: 10주에서 확인한 2편을 본문에 배치
3. **데이터와 방법 (1.5쪽)**: 데이터 출처, 표본, 표 1 (표본 특성), 분석 기법
4. **결과 (1쪽)**: 11주 1차 검정 + 이번 주 회귀 결과 요약

논의·한계·결론은 13주에 작성한다.

#### ④ 재현성 증빙

- 노트북 "Restart & Run All" 실행 스크린샷
- 다른 학생 1명이 실행해도 오류 없이 완주됨을 확인 (확인자 이름 + 일시 기록)

### 흔한 오류 3가지

| 오류 | 확인법 |
|---|---|
| 이분형 종속변수에 OLS를 적용한다 | 종속변수 분포를 히스토그램으로 먼저 확인한다 |
| 다중공선성을 확인하지 않는다 | VIF를 반드시 계산하고 본문에 기록한다 |
| 통계적 유의성과 효과크기를 혼동한다 | "p < 0.05이지만 β가 매우 작다"는 상황도 서술한다 |

## 제출물 체크리스트

- [ ] 회귀 결과표 (β, SE, p, 95% CI) — 로지스틱은 오즈비 포함
- [ ] 잔차 진단 또는 적합도 결과
- [ ] VIF (OLS일 경우)
- [ ] 본문 초안 4쪽 (서론·선행연구·방법·결과)
- [ ] 재현성 확인자 1명의 이름 + 실행 일시
- [ ] 노트북 "Restart & Run All" 스크린샷

## Exit Ticket

> **Q.** 본인의 회귀 결과에서 가장 예상 밖이었던 계수는 무엇이며, 이를 어떻게 해석하는가? (1-2문장)

**다음 시간 예고**: 13주차에는 논의·한계·결론을 완성하여 전체 초안(6-8쪽)을 마무리하고, 동료 교차리뷰와 발표 리허설을 진행한다.